# Evaluation — ArtBench-10

Este notebook serve para avaliar modelos generativos no ArtBench-10.

Funcionalidades:
- carregar o dataset com o helper do professor;
- recolher **5000 imagens reais**;
- gerar **5000 imagens falsas**;
- calcular **FID** e **KID**;
- repetir para várias **seeds/checkpoints**;
- guardar resultados em CSV e JSON.

Este notebook foi feito para ser reutilizável com:
- **GAN**
- **VAE**

> Antes de correr:
> 1. confirma os caminhos;
> 2. ajusta `MODEL_TYPE`;
> 3. indica os checkpoints corretos.

## 1. Instalar dependências

In [ ]:
# Descomenta se ainda não tiveres isto instalado
# !pip install torchmetrics[image]

## 2. Imports

In [ ]:
from __future__ import annotations

import sys
import csv
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid, save_image

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

## 3. Configuração

In [ ]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 128
NUM_WORKERS = 0
LATENT_DIM = 100
N_EVAL_SAMPLES = 5000

# escolhe: "gan" ou "vae"
MODEL_TYPE = "gan"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("MODEL_TYPE:", MODEL_TYPE)

## 4. Caminhos

In [ ]:
# Ajusta estes caminhos se necessário
PROJECT_ROOT = Path("..")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
KAGGLE_ROOT = PROJECT_ROOT.parent / "ArtBench-10"
TRAINING_CSV_PATH = PROJECT_ROOT / "training_20_percent.csv"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results" / "metrics"
SAMPLES_DIR = PROJECT_ROOT / "results" / "samples_eval"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT.resolve())
print("SCRIPTS_DIR  =", SCRIPTS_DIR.resolve())
print("KAGGLE_ROOT  =", KAGGLE_ROOT.resolve())
print("CHECKPOINTS  =", CHECKPOINT_DIR.resolve())

assert SCRIPTS_DIR.exists(), f"Não existe: {SCRIPTS_DIR.resolve()}"
assert KAGGLE_ROOT.exists(), f"Não existe: {KAGGLE_ROOT.resolve()}"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

## 5. Dataset helper do professor

In [ ]:
from artbench_local_dataset import load_kaggle_artbench10_splits

hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)

print("Train size:", len(train_hf))
print("Número de classes:", len(class_names))
print("Classes:", class_names)

## 6. Transform e dataset PyTorch

In [ ]:
transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex["image"]
        y = int(ex["label"])
        x = self.transform(img) if self.transform is not None else img
        return x, y, real_idx

def denorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)

## 7. Escolher subset ou full train

In [ ]:
USE_SUBSET_20 = True
INDEX_COLUMN = "train_id_original"

def load_ids_from_training_csv(csv_path: Path, index_column: str = "train_id_original") -> list[int]:
    ids = []
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if index_column not in (reader.fieldnames or []):
            raise ValueError(f"Coluna {index_column!r} não encontrada. Disponíveis: {reader.fieldnames}")
        for row in reader:
            value = str(row.get(index_column, "")).strip()
            if value:
                ids.append(int(value))
    if len(ids) == 0:
        raise ValueError("Não foram lidos IDs do CSV.")
    return ids

if USE_SUBSET_20:
    eval_indices = load_ids_from_training_csv(TRAINING_CSV_PATH, INDEX_COLUMN)
    print("A usar subset de 20% com", len(eval_indices), "imagens.")
else:
    eval_indices = None
    print("A usar conjunto completo de treino.")

eval_ds = HFDatasetTorch(train_hf, transform=transform, indices=eval_indices)
eval_loader = DataLoader(
    eval_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Dataset length:", len(eval_ds))
print("Batches:", len(eval_loader))

## 8. Ver algumas imagens reais

In [ ]:
x, y, idx = next(iter(eval_loader))
grid = make_grid(denorm(x[:36]), nrow=6, padding=2)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Imagens reais usadas na avaliação")
plt.show()

## 9. Arquiteturas suportadas: GAN e VAE

In [ ]:
class Generator32(nn.Module):
    def __init__(self, latent_dim=100, base_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, base_ch * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),

            nn.ConvTranspose2d(base_ch, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(128 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(128 * 4 * 4, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, 128 * 4 * 4)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x).view(x.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        h = self.decoder_fc(z).view(z.size(0), 128, 4, 4)
        recon = self.decoder(h)
        return recon, mu, logvar

    def sample(self, n, device):
        z = torch.randn(n, self.latent_dim, device=device)
        h = self.decoder_fc(z).view(n, 128, 4, 4)
        return self.decoder(h)

## 10. Indicar checkpoints a avaliar

In [ ]:
# Exemplos:
# GAN:
# CHECKPOINTS = {
#     42: CHECKPOINT_DIR / "dcgan_generator_subset20_seed42.pth",
#     43: CHECKPOINT_DIR / "dcgan_generator_subset20_seed43.pth",
# }
#
# VAE:
# CHECKPOINTS = {
#     42: CHECKPOINT_DIR / "vae_subset20_seed42.pth",
#     43: CHECKPOINT_DIR / "vae_subset20_seed43.pth",
# }

CHECKPOINTS = {
    42: CHECKPOINT_DIR / "dcgan_generator_subset20.pth",
}

for seed, ckpt in CHECKPOINTS.items():
    print(seed, "->", ckpt)

## 11. Funções de carregamento do modelo

In [ ]:
def load_model_for_evaluation(model_type: str, checkpoint_path: Path, device: torch.device):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint não encontrado: {checkpoint_path}")

    if model_type == "gan":
        model = Generator32(latent_dim=LATENT_DIM).to(device)
        state = torch.load(checkpoint_path, map_location=device)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        model.load_state_dict(state)
        model.eval()
        return model

    elif model_type == "vae":
        model = VAE(latent_dim=128).to(device)
        state = torch.load(checkpoint_path, map_location=device)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        model.load_state_dict(state)
        model.eval()
        return model

    else:
        raise ValueError("MODEL_TYPE tem de ser 'gan' ou 'vae'")

## 12. Recolher 5000 imagens reais

In [ ]:
@torch.no_grad()
def collect_real_images(dataloader, n_samples=5000):
    real_batches = []
    total = 0

    for x, _, _ in dataloader:
        x = denorm(x.cpu())
        real_batches.append(x)
        total += x.size(0)
        if total >= n_samples:
            break

    real_images = torch.cat(real_batches, dim=0)[:n_samples]
    return real_images

real_images = collect_real_images(eval_loader, n_samples=N_EVAL_SAMPLES)
print("Real images shape:", real_images.shape)

## 13. Gerar 5000 imagens falsas

In [ ]:
@torch.no_grad()
def generate_fake_images(model, model_type="gan", n_samples=5000, batch_size=128, latent_dim=100, device="cuda"):
    fake_batches = []
    total = 0

    while total < n_samples:
        current_batch = min(batch_size, n_samples - total)

        if model_type == "gan":
            z = torch.randn(current_batch, latent_dim, 1, 1, device=device)
            fake = model(z)

        elif model_type == "vae":
            fake = model.sample(current_batch, device=device)

        else:
            raise ValueError("model_type tem de ser 'gan' ou 'vae'")

        fake = denorm(fake.cpu())
        fake_batches.append(fake)
        total += current_batch

    fake_images = torch.cat(fake_batches, dim=0)
    return fake_images

## 14. Calcular FID e KID

In [ ]:
def to_uint8(x):
    return (x * 255).clamp(0, 255).to(torch.uint8)

def compute_fid_kid(real_images, fake_images, device):
    real_uint8 = to_uint8(real_images)
    fake_uint8 = to_uint8(fake_images)

    fid = FrechetInceptionDistance(feature=2048).to(device)
    kid = KernelInceptionDistance(subset_size=100).to(device)

    fid.update(real_uint8.to(device), real=True)
    fid.update(fake_uint8.to(device), real=False)
    fid_value = fid.compute().item()

    kid.update(real_uint8.to(device), real=True)
    kid.update(fake_uint8.to(device), real=False)
    kid_mean, kid_std = kid.compute()

    return {
        "fid": fid_value,
        "kid_mean": kid_mean.item(),
        "kid_std": kid_std.item()
    }

## 15. Testar uma seed/checkpoint

In [ ]:
# escolhe uma seed que exista no dicionário CHECKPOINTS
test_seed = list(CHECKPOINTS.keys())[0]
test_ckpt = CHECKPOINTS[test_seed]

model = load_model_for_evaluation(MODEL_TYPE, test_ckpt, device)
fake_images = generate_fake_images(
    model,
    model_type=MODEL_TYPE,
    n_samples=N_EVAL_SAMPLES,
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
    device=device,
)

print("Fake images shape:", fake_images.shape)

metrics = compute_fid_kid(real_images, fake_images, device)
print("Metrics:", metrics)

## 16. Guardar algumas amostras qualitativas

In [ ]:
seed_samples_dir = SAMPLES_DIR / f"{MODEL_TYPE}_seed_{test_seed}"
seed_samples_dir.mkdir(parents=True, exist_ok=True)

# guardar 32 imagens individuais
for i, img in enumerate(fake_images[:32]):
    save_image(img, seed_samples_dir / f"sample_{i:03d}.png")

# guardar grelha
grid = make_grid(fake_images[:64], nrow=8, padding=2)
save_image(grid, seed_samples_dir / "grid.png")

plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title(f"Amostras qualitativas - {MODEL_TYPE} - seed {test_seed}")
plt.show()

## 17. Avaliar todas as seeds/checkpoints

In [ ]:
all_results = []

for seed, ckpt in CHECKPOINTS.items():
    print(f"\nA avaliar seed {seed}...")
    model = load_model_for_evaluation(MODEL_TYPE, ckpt, device)

    fake_images = generate_fake_images(
        model,
        model_type=MODEL_TYPE,
        n_samples=N_EVAL_SAMPLES,
        batch_size=BATCH_SIZE,
        latent_dim=LATENT_DIM,
        device=device,
    )

    metrics = compute_fid_kid(real_images, fake_images, device)
    metrics["seed"] = seed
    metrics["checkpoint"] = str(ckpt)
    all_results.append(metrics)

    print(metrics)

results_df = pd.DataFrame(all_results)
results_df

## 18. Resumo final: média e desvio padrão

In [ ]:
summary = {
    "model_type": MODEL_TYPE,
    "n_runs": int(len(results_df)),
    "fid_mean": float(results_df["fid"].mean()),
    "fid_std": float(results_df["fid"].std(ddof=1)) if len(results_df) > 1 else 0.0,
    "kid_mean_mean": float(results_df["kid_mean"].mean()),
    "kid_mean_std": float(results_df["kid_mean"].std(ddof=1)) if len(results_df) > 1 else 0.0,
    "kid_std_mean": float(results_df["kid_std"].mean()),
}

summary

## 19. Guardar resultados

In [ ]:
results_csv = RESULTS_DIR / f"{MODEL_TYPE}_evaluation_results.csv"
summary_json = RESULTS_DIR / f"{MODEL_TYPE}_evaluation_summary.json"

results_df.to_csv(results_csv, index=False)

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

print("Resultados guardados em:")
print(results_csv.resolve())
print(summary_json.resolve())

## 20. Próximas notas

- **Sim, este notebook também pode ser usado para o VAE.**
- O que muda é:
  - `MODEL_TYPE = "vae"`
  - os ficheiros em `CHECKPOINTS`
  - a arquitetura do VAE tem de ser compatível com os pesos guardados

Se o teu VAE tiver uma arquitetura diferente desta, tens de substituir a classe `VAE` neste notebook pela tua versão exata.